# 01 · Bleed Anatomy — what ε-bleed does to a target, and why the smallest losses are cleanest

**How much does contaminating the training target with a fraction ε of the accompaniment
actually cost a compact separator — and can trimming the highest-loss chunks buy it back?**
This notebook is the visual companion to [`../THEORY.md`](../THEORY.md): it shows what the
ε-bleed corruption looks like, demonstrates the mixture-invariance property, pictures the
corrupted-optimal mask (which leaks ε·a *by construction*), renders the closed-form
**prediction line** on synthetic stems, and previews the trimming mechanism on a constructed
batch (which chunks get dropped, and why).

Nothing here needs a GPU. Most cells run on CPU from synthetic stems + the `singnet`
functions alone; the two that need a decoded MUSDB shard (the real spectrogram/audio
overlays) are marked **⚠️ RUN THIS LATER**. The notebook ships **un-executed** so the
committed file is a clean scaffold.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §3.1 (the corruption model),
  §3.2 (the mitigation); [`../THEORY.md`](../THEORY.md) §2 (corrupted-optimal model),
  §3 (the prediction line — the crux), §5 (the selection mechanism — the crux).
- **Data prep is *not* repeated here.** Acquisition, licensing, the 86/14/50 split, and
  the STFT front end live in Direction 01's notebooks
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb)).
  This direction reuses them verbatim and corrupts only the **training** targets.
- **Code, not prose, is authoritative:** corruption is `singnet.data.StemBleed`; the
  prediction line is `singnet.analysis.bleed`; the trimmed loss is
  `singnet.losses.TrimmedLoss`. Every number below is asserted in `tests/` (gate G0).

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · The corruption model and its mixture invariance

`StemBleed(ε)` redistributes a fraction ε of a track's accompaniment into its vocal
**target**: $\tilde v = v + \varepsilon a$, $\tilde a = (1-\varepsilon)a$. The defining
property (MASTER_PLAN §3.1, THEORY §1): the corrupted stems still sum to the **true
mixture**, $\tilde v + \tilde a = v + a = x$, so the network *input* is untouched — only
the supervised target is corrupted. The transform is deterministic (no RNG), so it cannot
perturb any augmentation stream.

In [ ]:
# CPU-runnable now: the exact formulas + mixture invariance on synthetic stems.
import numpy as np
from singnet.data import StemBleed

rng = np.random.default_rng(0)
v = rng.standard_normal(8).astype('float32')
a = rng.standard_normal(8).astype('float32')
for eps in (0.05, 0.15, 0.30):
    out = StemBleed(eps)({'vocals': v, 'accompaniment': a})
    vt, at = out['vocals'], out['accompaniment']
    inv = np.max(np.abs((vt + at) - (v + a)))
    print(f'ε={eps:>4}:  ṽ=v+εa OK={np.allclose(vt, v+eps*a)}  '
          f'ã=(1−ε)a OK={np.allclose(at, (1-eps)*a)}  '
          f'‖(ṽ+ã)−(v+a)‖∞={inv:.2e}  (mixture untouched)')

## 2 · What ε-bleed looks like (waveform overlay)

At each ε the vocal target gains a scaled copy of the accompaniment. On synthetic tones the
overlay makes the redistribution visible; the real MUSDB waveform/spectrogram overlay needs
one decoded shard and is **RUN LATER**.

In [ ]:
# CPU-runnable now: synthetic vocal + accompaniment tones, target overlay at each ε.
import numpy as np
import matplotlib.pyplot as plt
from singnet.data import StemBleed

sr = 44100
t = np.arange(int(0.02 * sr)) / sr
v = (0.3 * np.sin(2 * np.pi * 440 * t)).astype('float32')          # vocal tone
a = (0.3 * np.sin(2 * np.pi * 130 * t)).astype('float32')          # accompaniment tone

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t * 1000, v, label='clean vocal target v', lw=1.5, color='#2b6cb0')
for eps, col in ((0.05, '#f6c85f'), (0.15, '#e07b39'), (0.30, '#c0392b')):
    vt = StemBleed(eps)({'vocals': v, 'accompaniment': a})['vocals']
    ax.plot(t * 1000, vt, label=f'ṽ = v + {eps}·a', lw=1, alpha=0.8, color=col)
ax.set_xlabel('time (ms)'); ax.set_ylabel('amplitude')
ax.set_title('ε-bleed corrupts the vocal target (input mixture unchanged)')
ax.legend(fontsize=8, ncol=2); fig.tight_layout(); plt.show()

In [ ]:
# ⚠️ RUN THIS LATER (CPU, needs one decoded MUSDB shard) — real spectrogram + audio overlay.
# A vocal stem's log-magnitude spectrogram, clean vs ṽ=v+εa at ε∈{0.05,0.15,0.30}, showing
# the accompaniment energy leaking into the "vocals" target; plus IPython.display.Audio cells.
#
#   from singnet.data import WavShardStore, StemBleed
#   from singnet.audio import STFT, analyze_chunk
#   store = WavShardStore(os.environ['SHARD_ROOT']); src = store.load_sources('<track>')
#   # slice a 6 s window, apply StemBleed(ε), STFT both, imshow log-magnitude side by side,
#   # and display Audio(v, rate=sr) vs Audio(v+ε*a, rate=sr) to *hear* the bleed.
print('Real spectrogram/audio overlay — RUN LATER (needs a decoded shard).')

## 3 · The corrupted-optimal mask leaks ε·a (THEORY §2)

For L1-magnitude training the per-bin optimum is
$M^\star = \operatorname{clip}(|V+\varepsilon A|/|V+A|,\,0,\,1)$, so the masked estimate
magnitude reproduces the **corrupted-target** magnitude $|V+\varepsilon A|$ — the model is
trained to keep ε·a in its output. The cell shows, on a synthetic magnitude spectrum, that
the optimal masked magnitude tracks $|V+\varepsilon A|$ (not the clean $|V|$): a leak of
size O(ε) by construction. The mask also wears the *mixture* phase (an ε-independent
penalty, THEORY §2.2), which is why the dose–response **curve** — a difference — is the
clean observable.

In [ ]:
# CPU-runnable now: the L1-optimal masked magnitude reproduces |V+εA|, i.e. it leaks ε·a.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
F = 64
V = (rng.standard_normal(F) + 1j * rng.standard_normal(F))          # clean vocal spectrum
A = (rng.standard_normal(F) + 1j * rng.standard_normal(F))          # accompaniment spectrum
X = V + A                                                            # mixture (untouched)
eps = 0.30
tgt = np.abs(V + eps * A)                                            # corrupted-target magnitude
Mstar = np.clip(tgt / np.abs(X), 0.0, 1.0)                           # per-bin L1 optimum (THEORY §2.1)
est = Mstar * np.abs(X)                                              # masked estimate magnitude

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(np.abs(V), label='clean target |V|', lw=1.4, color='#2b6cb0')
ax.plot(tgt, label='corrupted target |V+εA|', lw=1.2, color='#c0392b')
ax.plot(est, '--', label='optimal masked mag M*·|X|', lw=1.0, color='#2b6cb0', alpha=0.7)
ax.set_title(f'L1-optimal masked magnitude tracks |V+εA| (ε={eps}) — leaks ε·a')
ax.set_xlabel('frequency bin'); ax.set_ylabel('magnitude'); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()
print('mean |M*·|X| − |V+εA||  (≈0, clip inactive):', float(np.mean(np.abs(est - tgt))))
print('mean |M*·|X| − |V||     (the O(ε) leak)   :', float(np.mean(np.abs(est - np.abs(V)))))

## 4 · The prediction line P(ε) on synthetic stems (THEORY §3)

A model that perfectly learns the corrupted target outputs $\hat v = v + \varepsilon a$; its
**clean** SI-SDR is $P(\varepsilon)=\text{SI-SDR}(v+\varepsilon a,\,v)$, computed by
`singnet.analysis.bleed.prediction_line` (exact projection form + the $v\perp a$ closed form
$10\log_{10}\!\frac{\lVert v\rVert^2}{\varepsilon^2\lVert a\rVert^2}$). On orthogonal stems
the two columns coincide and give the $-20\log_{10}\varepsilon$ line.

In [ ]:
# CPU-runnable now: prediction line on synthetic stems (orthogonal + random), both columns.
import numpy as np
import matplotlib.pyplot as plt
from singnet.analysis import prediction_line, predicted_curve

# an orthogonal pair (exact == closed form) and a few random tracks (a non-orthogonal cloud)
orth = ('orthogonal', np.array([1., 1., 1., 1.]), np.array([1., -1., 1., -1.]))
rng = np.random.default_rng(2)
rand = [(f'rand{i}', rng.standard_normal(2048), rng.standard_normal(2048)) for i in range(6)]
eps_grid = [0.02, 0.05, 0.10, 0.15, 0.30]

df = prediction_line([orth] + rand, eps_grid)
curve_exact = predicted_curve(df, column='si_sdr_exact')
curve_orth = predicted_curve(df, column='si_sdr_orth')

fig, ax = plt.subplots(figsize=(7, 4))
o = df[df['track'] == 'orthogonal']
ax.plot(o['epsilon'], o['si_sdr_exact'], 'o-', color='#2b6cb0', label='orthogonal track: exact = −20·log10(ε)')
ax.plot(curve_exact.index, curve_exact.values, 's--', color='#c0392b', label='P(ε): mean exact projection')
ax.plot(curve_orth.index, curve_orth.values, '^:', color='#e07b39', label='mean orthogonal approximation')
ax.set_xscale('log'); ax.set_xlabel('ε (log)'); ax.set_ylabel('predicted clean SI-SDR (dB)')
ax.set_title('Prediction line P(ε): a −20·log10(ε) null anchored by energy ratios')
ax.legend(fontsize=8); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()
print(df[df['track'] == 'orthogonal'][['epsilon', 'si_sdr_exact', 'si_sdr_orth', 'ortho_gap_db']].to_string(index=False))

## 5 · The trimming mechanism preview (THEORY §5)

Under **uniform** bleed every target is corrupted, so trimming cannot select "clean vs
corrupted." What it *can* do is select the chunks whose accompaniment is quiet — the
effectively cleanest targets — because per-chunk irreducible loss grows with accompaniment
energy (THEORY §5). On a constructed batch with known per-chunk losses (and energies that
track them), `TrimmedLoss` keeps the lowest-loss chunks; the testable signature is that the
**kept** chunks carry lower ⟨a⟩-energy than the **dropped** ones.

In [ ]:
# CPU-runnable now: which chunks get dropped, and the kept-vs-dropped energy split.
import torch
from singnet.losses import TrimmedLoss, build

B, F, T = 8, 4, 4
per_chunk_loss = [0.9, 0.1, 0.7, 0.2, 0.5, 0.3, 0.8, 0.4]   # a known per-chunk L1 ranking
mask = torch.zeros(B, F, T)                                  # est=0 -> per-chunk loss = |target|
mix = torch.ones(B, F, T)
tgt = torch.zeros(B, F, T)
for i, cval in enumerate(per_chunk_loss):
    tgt[i] = cval
energy = torch.tensor(per_chunk_loss)                        # ⟨a⟩-energy tracks loss (THEORY §5)

trimmed = TrimmedLoss(build('l1mag'), q=0.30)               # keep ⌈0.7·8⌉ = 6, drop 2
loss, aux = trimmed(mask, mix, tgt, chunk_energy=energy)
print(f'keep {int(aux["n_kept"])}/{B}  (kept_fraction={aux["kept_fraction"]:.3f})')
print('kept chunk indices   :', sorted(aux['kept_idx']))
print('dropped chunk indices:', sorted(aux['dropped_idx']), '(the highest-loss / loudest-⟨a⟩)')
print(f'kept ⟨a⟩-energy mean    = {aux["kept_energy_mean"]:.3f}')
print(f'dropped ⟨a⟩-energy mean = {aux["dropped_energy_mean"]:.3f}   (THEORY §5: kept < dropped)')

## 6 · Takeaways

- **Mixture invariance** holds to float round-off: the corruption moves accompaniment from
  the target's *complement* into the target, leaving the input mixture exactly $v+a$.
- The **corrupted-optimal** L1 mask reproduces $|V+\varepsilon A|$ — it leaks ε·a by
  construction (THEORY §2); the clean cost is isolated by the dose–response *curve*.
- The **prediction line** $P(\varepsilon)$ is a $-20\log_{10}\varepsilon$ null anchored per
  track; the measured curve's position *relative to* $P$ (above/on/below) is the payload
  (notebook 02's headline figure).
- Under uniform bleed, **trimming is curriculum-by-cleanliness**: it keeps the low-⟨a⟩-energy
  chunks. The kept < dropped energy split is the falsifiable §5 signature the training loop
  logs every 500 steps.